In [1]:
import pandas as pd
import re
from src.utils import *
import datetime as dt
import glob

In [2]:
input_file = 'Data/archive/ukb45839.tab'

In [3]:
column_names = pd.read_csv(input_file, sep="\t", nrows=0).columns

In [4]:
column_names[15028]

'f.41203.0.0'

In [5]:
#columns_to_read

In [6]:
_, _, set12eids, set3eids = get_data({'dset':'cmb', 'target':'mort', 'combine_sets':True})

In [12]:
# Read the data
case_cols = [f"{column_names[i]}" for i in range(14962, 15028)]
date_cols = [f"{column_names[i]}" for i in range(15717, 15783)]
columns_to_read = ["f.eid"] + case_cols + date_cols
icd10_all = pd.read_csv(input_file, sep="\t", usecols=columns_to_read, dtype=str)

KeyboardInterrupt: 

In [13]:
allsetseids = np.concatenate((set12eids, set3eids))

In [ ]:
icd10_all['f.eid'].astype('int')

In [ ]:
#icd10 = icd10_all[icd10_all['f.eid'].astype('int').isin(set3eids)]

In [ ]:
icd10 = icd10_all[icd10_all['f.eid'].astype('int').isin(allsetseids)]

In [ ]:
icd10.shape

In [13]:
# Define disease categories and corresponding ICD-10 codes as given by Tigist
diseases = {
    "IschemicHeartDisease": ["I20", "I21", "I22", "I23", "I24", "I25"],
    "Stroke": ["I60", "I61", "I62", "I63", "I64"],
    "HeartFailure": ["I50"],
    "PeripheralArterialDisease": ["I70", "I73"],
    "AtrialFibrillation": ["I48"],
}

def get_earliest_time (row):
    col_idx = row['col_idx']
    lowest_date = dt.date.today()
    
    date_str = "f.41262.0."

    for idx in col_idx:
        date = pd.to_datetime(row[date_str + str(idx)])
        if date.date() < lowest_date:
            date = lowest_date
            lowest_date_col = date_str + str(idx)
    return row[lowest_date_col]

# Define a function to process each disease
def process_disease(disease_name, icd_codes, df):
    # Combine ICD-10 codes into a regex pattern
    icd_pattern = "|".join(icd_codes)
    
    # Identify rows where any of the ICD columns match the codes
    icd_columns = [col for col in df.columns if col in case_cols]
    df_icd = df[df[icd_columns].apply(lambda row: row.str.contains(icd_pattern, na=False).any(), axis=1)]
    df_icd[disease_name] = 1

    #Save all columns that match icd pattern
    df_icdtrue = df_icd[icd_columns].apply(lambda row: row.str.contains(icd_pattern, na=False), axis=1)
    
    #Match date columns row wise 
    df_icd['col_idx'] = df_icdtrue.apply(lambda row: np.where(row.values==True)[0], axis=1)
    df_icd[f'{disease_name}_date'] = df_icd.apply(lambda row: get_earliest_time(row), axis =1)
    df_icd = df_icd[['f.eid', disease_name, f'{disease_name}_date']]

    # Save results for this disease
    output_file = f"Data/endpoints/merge_{disease_name}.date.csv"
    df_icd.to_csv(output_file, index=False)
    print(f"Processed {disease_name}: {len(df_icd)} rows saved to {output_file}")

# Process each disease
for disease, codes in diseases.items():
    process_disease(disease, codes, icd10)

print("Processing complete. Files saved in Data")


/tmp/ipykernel_3871776/3399946957.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_icd[disease_name] = 1
/tmp/ipykernel_3871776/3399946957.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_icd['col_idx'] = df_icdtrue.apply(lambda row: np.where(row.values==True)[0], axis=1)
/tmp/ipykernel_3871776/3399946957.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentatio

Processed IschemicHeartDisease: 3523 rows saved to Data/endpoints/merge_IschemicHeartDisease.date.csv


/tmp/ipykernel_3871776/3399946957.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_icd[disease_name] = 1
/tmp/ipykernel_3871776/3399946957.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_icd['col_idx'] = df_icdtrue.apply(lambda row: np.where(row.values==True)[0], axis=1)
/tmp/ipykernel_3871776/3399946957.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentatio

Processed Stroke: 1083 rows saved to Data/endpoints/merge_Stroke.date.csv


/tmp/ipykernel_3871776/3399946957.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_icd[disease_name] = 1
/tmp/ipykernel_3871776/3399946957.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_icd['col_idx'] = df_icdtrue.apply(lambda row: np.where(row.values==True)[0], axis=1)
/tmp/ipykernel_3871776/3399946957.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentatio

...

/tmp/ipykernel_3871776/3399946957.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_icd[disease_name] = 1
/tmp/ipykernel_3871776/3399946957.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_icd['col_idx'] = df_icdtrue.apply(lambda row: np.where(row.values==True)[0], axis=1)
/tmp/ipykernel_3871776/3399946957.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentatio

Processed PeripheralArterialDisease: 259 rows saved to Data/endpoints/merge_PeripheralArterialDisease.date.csv


/tmp/ipykernel_3871776/3399946957.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_icd[disease_name] = 1
/tmp/ipykernel_3871776/3399946957.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_icd['col_idx'] = df_icdtrue.apply(lambda row: np.where(row.values==True)[0], axis=1)


Processed AtrialFibrillation: 1374 rows saved to Data/endpoints/merge_AtrialFibrillation.date.csv
Processing complete. Files saved in Data


/tmp/ipykernel_3871776/3399946957.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_icd[f'{disease_name}_date'] = df_icd.apply(lambda row: get_earliest_time(row), axis =1)


In [9]:
file_pattern = "Data/endpoints/merge_*.date.csv"

cvd_dataframes = [pd.read_csv(file) for file in glob.glob(file_pattern)]

# Merge all dataframes on the 'f.eid' column
cvd_merged_df = cvd_dataframes[0]
for df in cvd_dataframes[1:]:
    cvd_merged_df = pd.merge(cvd_merged_df, df, on="f.eid", how="outer")
cvd_merged_df['eid'] = cvd_merged_df['f.eid']
cvd_merged_df['CVD_inc'] = 1
cvd_merged_df = cvd_merged_df.apply(lambda col: pd.to_datetime(col, errors="coerce") if col.name.endswith("date") else col)
cvd_merged_df['CVD_date'] = cvd_merged_df[[col for col in cvd_merged_df.columns if col.endswith("date")]].min(axis =1)
cvd_merged_df['CVD_date'] 
max_date = cvd_merged_df[[col for col in cvd_merged_df.columns if col.endswith("date")]].max().max()
cvd_merged_df = cvd_merged_df[['eid', 'CVD_inc', 'CVD_date']]
#cvd_merged_df

In [10]:
max_date

Timestamp('2020-11-30 00:00:00')

In [14]:
controls = pd.DataFrame(allsetseids, columns = ['eid'])
controls = controls[~controls['eid'].isin(cvd_merged_df['eid'])]
controls['CVD_date'] = max_date + pd.Timedelta(days=1)
controls['CVD_inc'] = 0

cvd_set = cvd_merged_df.merge(controls, how = 'outer')
#cvd_set.to_csv('Data/FollowUpCVD.csv', index = False)
cvd_set.shape

(40696, 3)

In [21]:
basicinfo = pd.read_csv("Data/covar/basicinfo_instance_0.csv")
basicinfo = basicinfo[basicinfo['eid'].isin(cvd_set['eid'])]
cvd_set_time = cvd_set.merge(basicinfo[['eid', 'date_center.0.0', 'age_center.0.0']], on = 'eid', how = 'inner')
cvd_set_time['CVD_followup'] = cvd_set_time['CVD_date'] - pd.to_datetime(cvd_set_time['date_center.0.0'])
cvd_set_time['CVD_prev'] = (cvd_set_time['CVD_followup'] < pd.Timedelta(0)).astype(int)
cvd_set_time['CVD_age'] = np.where(
    cvd_set_time['CVD_prev'] == 0, 
    (cvd_set_time['CVD_followup'].dt.days / 365.25) + cvd_set_time['age_center.0.0'], 
    pd.NA
)
cvd_set_time['CVD_inc'] = np.where(
    cvd_set_time['CVD_prev'] == 1, 
    pd.NA, cvd_set_time['CVD_inc']
)
cvd_set_time['CVD_prev'].sum()
cvd_set_time_save = cvd_set_time[['eid', 'CVD_prev','CVD_inc', 'CVD_age', 'CVD_followup']]
cvd_set_time_save.to_csv('Data/endpoints/FollowUpCVD.csv', index = False)

In [20]:
(cvd_set_time_save.loc[cvd_set_time_save['CVD_prev'] == 0, 'followup'].dt.days / 365.25).describe()

count    38656.000000
mean        11.350541
std          2.080418
min          0.005476
25%         11.017112
50%         11.728953
75%         12.476386
max         14.715948
Name: followup, dtype: float64

In [18]:
# ukb_outcomes = pd.read_csv('Data/ukb675807.csv', nrows=0).columns
# list(ukb_outcomes)

In [19]:
columns_to_read2 = ['eid', '46-0.0', '47-0.0', '2178-0.0'] 
ukb_outcomes = pd.read_csv('Data/archive/ukb675807.csv',usecols=columns_to_read2)

In [20]:
ukb_outcomes = ukb_outcomes[ukb_outcomes['eid'].astype('int').isin(allsetseids)]

In [21]:
ukb_outcomes['max_handgrip'] = ukb_outcomes[['46-0.0', '47-0.0']].max(axis=1)
ukb_outcomes['self_perceived_health'] = ukb_outcomes['2178-0.0']
ukb_outcomes_keep = ukb_outcomes[['eid', 'max_handgrip', 'self_perceived_health']]
ukb_outcomes_keep.loc[:,'self_perceived_health'] = np.where(ukb_outcomes_keep['self_perceived_health'] < 0, np.nan, ukb_outcomes_keep['self_perceived_health'])
ukb_outcomes_keep.loc[:,'self_perceived_health'] = ukb_outcomes_keep['self_perceived_health'].replace({4: 0, 3: 1, 2: 2, 1: 3})

ukb_outcomes_keep.to_csv('Data/endpoints/HandgripSelfperceived.csv', index=False)

In [22]:
ukb_outcomes_keep.describe()

,eid,max_handgrip,self_perceived_health
count,4.069300e+04,40572.000000,40396.000000
mean,3.512426e+06,32.458050,1.830231
std,1.454134e+06,11.317143,0.749662
min,1.000041e+06,0.000000,0.000000
25%,2.254806e+06,24.000000,1.000000
50%,3.513137e+06,30.000000,2.000000
75%,4.775023e+06,40.000000,2.000000
max,6.024800e+06,89.000000,3.000000


In [23]:
cancer.columns

NameError: name 'cancer' is not defined

In [ ]:
cancer_all = pd.read_csv('Data/endpoints/CancerDates.csv')

In [ ]:
cancer_all
cancer2 = cancer_all[cancer_all['eid'].isin(allsetseids)]
cancer2.shape

In [ ]:
cancer = pd.read_csv('Data/endpoints/CancerDates.csv')
cancer['cancer_date'] = pd.to_datetime(cancer['p40005_i0'], errors='coerce')
cancer['cancer_inc'] = (cancer['cancer_date'].notna()).astype(int)
max_date = cancer['cancer_date'].max() + pd.Timedelta(days=1)
cancer['cancer_date'].fillna(max_date, inplace=True)
cancer = cancer[cancer['eid'].isin(all)]
cancer = cancer[['eid', 'cancer_inc', 'cancer_date']]

cancer_time = cancer.merge(basicinfo[['eid', 'date_center.0.0', 'age_center.0.0']], on = 'eid', how = 'inner')
cancer_time['followup'] = cancer_time['cancer_date'] - pd.to_datetime(cancer_time['date_center.0.0'])
cancer_time['cancer_prev'] = (cancer_time['followup'] < pd.Timedelta(0)).astype(int)
cancer_time['cancer_age'] = np.where(
    cancer_time['cancer_prev'] == 0, 
    (cancer_time['followup'].dt.days / 365.25) + cancer_time['age_center.0.0'], 
    pd.NA
)
cancer_time['cancer_inc'] = np.where(
    cancer_time['cancer_prev'] == 1, 
    pd.NA, cancer_time['cancer_inc']
)
cancer_time['cancer_prev'].sum()
cancer_time_save = cancer_time[['eid', 'cancer_prev','cancer_inc', 'cancer_age']]
cancer_time_save.to_csv('Data/endpoints/FollowUpCancer.csv', index = False)
cancer_time_save

In [ ]:
cancer_time['cancer_date'].isna().sum()

In [ ]:
_, _, _, _, _, _, cols = get_data({'dset':'allprot', 'target':'mort'})

In [ ]:
cols.has_duplicates

In [ ]:
# icd_pattern = "|".join(["I20", "I21", "I22", "I23", "I24", "I25"])
# icd_columns = [col for col in IHD.columns if col in case_cols]
# IHDtrue = IHD[icd_columns].apply(lambda row: row.str.contains(icd_pattern, na=False), axis=1)
# IHD['col_idx'] = IHDtrue.apply(lambda row: np.where(row.values==True)[0], axis=1)

# import datetime as dt

# def get_earliest_time (row):
#     col_idx = row['col_idx']
#     lowest_date = dt.date.today()
    
#     date_str = "f.41262.0."

#     for idx in col_idx:
#         date = pd.to_datetime(row[date_str + str(idx)])
#         if date.date() < lowest_date:
#             date = lowest_date
#             lowest_date_col = date_str + str(idx)
#     return row[lowest_date_col]

In [ ]:
# get_earliest_time(IHD.iloc[0,])

In [ ]:
# IHD.apply(lambda row: get_earliest_time(row), axis =1)

In [ ]:
# IHD = pd.read_csv('merge_IschemicHeartDisease.date.csv')
# IHD.head()